# Multi-Backbone AI Evaluation

Evaluates GPT-4o judge scores across 4 backbones × 4 conditions.
Data loaded from `dissonance/multibackbone/{model}/{condition}/`

In [ ]:
# === Setup: Imports, API Key, Prompts ===
import os, json, re, time
from pathlib import Path
from collections import defaultdict
from openai import OpenAI
import pandas as pd
from tqdm.notebook import tqdm
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
EVALUATION_MODEL = "gpt-4o"
print(f"Evaluator: {EVALUATION_MODEL}")

# Evaluation prompts (MIRROR framework)
SYSTEM_PROMPT = """You are an expert psychological evaluator specializing in Cognitive Behavioral Therapy (CBT) and therapeutic alliance. Your task is to analyze a full counseling session transcript and score the therapist's performance based on the evaluation frameworks used in the MIRROR paper (arXiv:2504.13211v2).

CRITICAL EVALUATION FOCUS:
- You must assess the therapist's ability to perceive "Emotional Subtext"—the underlying feelings that may not be explicitly stated in the client's words but are hinted at through context, tone, or provided emotional metadata.
- A high-performing therapist should move beyond surface-level reflection and identify "Latent Concerns" (e.g., hidden guilt, worthiness issues, or conflicting emotions) to facilitate deeper discovery.

You must evaluate the entire session as a whole, not just individual turns.
"""

USER_PROMPT = """Please evaluate this therapy session. Pay close attention to how the therapist navigates the gap between what the client SAYS and what the client FEELS

[Session Transcript]
{CONVERSATION_TEXT_GOES_HERE}
[/Session Transcript]

1. **Therapist Skills (0-6):** Understanding, Interpersonal Effectiveness
2. **Client Alliance (1-5):** Affective Bond
3. **CTRS (0-6 each):** Collaboration, Guided Discovery, Focus, Strategy

Return EXACT JSON:
{{
  "therapist_skills": {{"understanding": 0, "interpersonal_effectiveness": 0}},
  "client_alliance": {{"affective_bond": 0}},
  "ctrs": {{"collaboration": 0, "guided_discovery": 0, "focus": 0, "strategy": 0}},
  "reasoning": "...",
  "comparative_advantage": "..."
}}
"""
print("Setup complete.")

In [ ]:
# === Data Loading ===
BASE = Path(r"C:\Luna-AI-Therapist\dissonance\multibackbone")

BACKBONES = ["gpt4o-mini", "claude", "qwen", "deepseek"]
CONDITIONS = {
    "baseline":    {"prefixes": ["baseline"],          "suffix": "full_baseline", "is_baseline": True},
    "emotion":     {"prefixes": ["emotion"],           "suffix": "full_emotion_online", "is_baseline": False},
    "multimodal":  {"prefixes": ["multimodal", "dissonance"], "suffix": "full_multimodal", "is_baseline": False},
    "dissonance":  {"prefixes": ["dissonance"],        "suffix": "full_dissonance_online", "is_baseline": False},
}

def extract_dialogue_id(p):
    m = re.search(r"(\d+)_full", p.name)
    return int(m.group(1)) if m else None

def load_dialogue_folder(folder, prefixes, suffix, is_baseline):
    dialogues = {}
    seen = set()
    for prefix in prefixes:
        for ext in [".jsonl", ".json"]:
            for path in sorted(folder.glob(f"{prefix}_*_{suffix}{ext}")):
                did = extract_dialogue_id(path)
                if did is None or did in seen: continue
                seen.add(did)
        if did is None: continue
        turns = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                rec = json.loads(line)
                turns.append({
                    "transcript": rec["client"],
                    ("therapist_response_baseline" if is_baseline else "therapist_response"): rec["therapist"],
                })
        dialogues[did] = turns
    return dialogues

# Load all backbones × conditions
all_data = {}
for bb in BACKBONES:
    all_data[bb] = {}
    for cond, cfg in CONDITIONS.items():
        folder = BASE / bb / cond
        if not folder.exists():
            print(f"  WARNING: {folder} not found")
            all_data[bb][cond] = {}
            continue
        all_data[bb][cond] = load_dialogue_folder(folder, cfg["prefixes"], cfg["suffix"], cfg["is_baseline"])
        print(f"  {bb}/{cond}: {len(all_data[bb][cond])} dialogues")

# Find shared dialogue IDs across all loaded conditions
dialogue_ids = sorted(set.intersection(*[
    set(d.keys()) for bb in BACKBONES for d in [all_data[bb].get(c, {}) for c in CONDITIONS] if d
]))
print(f"\nShared dialogue IDs: {len(dialogue_ids)}")
print(f"Num dialogues to evaluate: {len(dialogue_ids)}")

In [ ]:
# === Session Formatting ===
def format_conversation_text(turns_list, is_baseline=False):
    full_text = ""
    tkey = "therapist_response_baseline" if is_baseline else "therapist_response"
    for turn in turns_list:
        ct = turn.get("transcript") or turn.get("client", "[missing]")
        tt = turn.get(tkey) or turn.get("therapist", "[missing]")
        full_text += f"CLIENT: {ct}\n\nTHERAPIST: {tt}\n\n"
    return full_text.strip()

# === GPT-4o Evaluation ===
def evaluate_session(session_text, max_retries=3):
    formatted = USER_PROMPT.format(CONVERSATION_TEXT_GOES_HERE=session_text)
    messages = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":formatted}]
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=EVALUATION_MODEL, messages=messages,
                temperature=0.0, response_format={"type":"json_object"})
            return json.loads(completion.choices[0].message.content)
        except Exception as e:
            print(f"    Attempt {attempt+1}/{max_retries}: {e}")
            if attempt < max_retries-1: time.sleep(5)
    return {"error": f"Failed after {max_retries}"}

print("Functions defined.")

In [ ]:
# === Evaluation Loop (all backbones × all conditions × shared dialogue IDs) ===
eval_scores = {bb: {c: [] for c in CONDITIONS} for bb in BACKBONES}
total = len(dialogue_ids) * len(BACKBONES) * len(CONDITIONS)
print(f"Total evaluations: {total}")

count = 0
for did in tqdm(dialogue_ids, desc="Evaluating"):
    for bb in BACKBONES:
        for cond, cfg in CONDITIONS.items():
            if did not in all_data[bb].get(cond, {}):
                continue
            session = format_conversation_text(all_data[bb][cond][did], is_baseline=(cond=="baseline"))
            scores = evaluate_session(session)
            scores["dialogue_id"] = did
            scores["method"] = cond
            scores["backbone"] = bb
            eval_scores[bb][cond].append(scores)
            count += 1
            u = scores.get("therapist_skills",{}).get("understanding","?")
            if count % 40 == 0:
                print(f"  [{count}/{total}] {bb}/{cond} dlg{did}: U={u}")

print(f"\nEvaluation complete: {count}/{total} sessions")

In [ ]:
# === Save Results ===
OUTPUT_DIR = Path("evaluation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

for bb in BACKBONES:
    for cond in CONDITIONS:
        path = OUTPUT_DIR / f"ai_eval_multibackbone_{bb}_{cond}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for s in eval_scores[bb][cond]:
                f.write(json.dumps(s, ensure_ascii=False) + "\n")
        print(f"  Saved {len(eval_scores[bb][cond])} scores -> {path}")

In [ ]:
# === Calculate Averages ===
def calc_avg(scores):
    df = pd.json_normalize(scores)
    if 'error' in df.columns: df = df[df['error'].isnull()]
    if df.empty:
        return {"understanding_avg":0,"interpersonal_effectiveness_avg":0,"affective_bond_avg":0,
                "ctrs_collab_avg":0,"ctrs_guided_avg":0,"ctrs_focus_avg":0,"ctrs_strategy_avg":0,"count":0}
    return {
        "understanding_avg": df['therapist_skills.understanding'].mean(),
        "interpersonal_effectiveness_avg": df['therapist_skills.interpersonal_effectiveness'].mean(),
        "affective_bond_avg": df['client_alliance.affective_bond'].mean(),
        "ctrs_collab_avg": df['ctrs.collaboration'].mean(),
        "ctrs_guided_avg": df['ctrs.guided_discovery'].mean(),
        "ctrs_focus_avg": df['ctrs.focus'].mean(),
        "ctrs_strategy_avg": df['ctrs.strategy'].mean(),
        "count": len(df),
    }

avgs = {bb: {c: calc_avg(eval_scores[bb][c]) for c in CONDITIONS} for bb in BACKBONES}
print("Averages calculated.")

In [ ]:
# === Cross-Backbone Comparison: Dissonance-Aware only ===
print("\n=== DISSONANCE-AWARE Comparison ===")
diss_data = {
    "Backbone": BACKBONES,
    "Understanding": [f"{avgs[bb]['dissonance']['understanding_avg']:.2f}" for bb in BACKBONES],
    "Interpersonal": [f"{avgs[bb]['dissonance']['interpersonal_effectiveness_avg']:.2f}" for bb in BACKBONES],
    "Affective Bond": [f"{avgs[bb]['dissonance']['affective_bond_avg']:.2f}" for bb in BACKBONES],
    "CTRS Collab": [f"{avgs[bb]['dissonance']['ctrs_collab_avg']:.2f}" for bb in BACKBONES],
    "CTRS Guided": [f"{avgs[bb]['dissonance']['ctrs_guided_avg']:.2f}" for bb in BACKBONES],
    "CTRS Focus": [f"{avgs[bb]['dissonance']['ctrs_focus_avg']:.2f}" for bb in BACKBONES],
    "CTRS Strategy": [f"{avgs[bb]['dissonance']['ctrs_strategy_avg']:.2f}" for bb in BACKBONES],
    "Succ": [f"{avgs[bb]['dissonance']['count']} / {len(dialogue_ids)}" for bb in BACKBONES],
}
display(pd.DataFrame(diss_data))

In [ ]:
# === Per-Backbone Tables (4 tables) ===
metrics = [
    ("Understanding (0-6)", "understanding_avg"),
    ("Interpersonal (0-6)", "interpersonal_effectiveness_avg"),
    ("Affective Bond (1-5)", "affective_bond_avg"),
    ("CTRS Collab (0-6)", "ctrs_collab_avg"),
    ("CTRS Guided (0-6)", "ctrs_guided_avg"),
    ("CTRS Focus (0-6)", "ctrs_focus_avg"),
    ("CTRS Strategy (0-6)", "ctrs_strategy_avg"),
    ("Successful", "count"),
]

for bb in BACKBONES:
    print(f"\n=== {bb.upper()} ===")
    data = {"Metric": [m[0] for m in metrics]}
    for cond in ["baseline","emotion","multimodal","dissonance"]:
        n = len(dialogue_ids)
        data[cond] = [f"{avgs[bb][cond][m[1]]:.2f}" if m[1] != "count" else f"{avgs[bb][cond]['count']} / {n}" for m in metrics]
    display(pd.DataFrame(data))

In [ ]:
# === Qualitative Feedback ===
for bb in BACKBONES:
    print(f"\n=== {bb.upper()} Qualitative ===")
    for cond in CONDITIONS:
        scores = eval_scores[bb][cond]
        print(f"\n--- {bb}/{cond} ---")
        for s in scores[:3]:
            print(f"  Dlg {s.get('dialogue_id','?')}: {s.get('reasoning','N/A')[:120]}...")
    print("-" * 60)